In [11]:
import os
import mlflow
import dagshub
from src.config import SEED, TARGET, TIME_COL, DAGSHUB_USERNAME,DAGSHUB_REPO , processed_dataset, src_path
from mlflow.tracking import MlflowClient




In [ ]:
dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True
)
client = MlflowClient()

In [2]:
LOCAL_LGBM_DIR = "models/lgbm_folds"
os.makedirs(LOCAL_LGBM_DIR, exist_ok=True)

# Replace with the Parent Run ID of your LightGBM 5-fold training experiment
PARENT_RUN_ID = "d8e6b2401c384292937b103c9f50fffe"

In [6]:
import mlflow.lightgbm

exp_id = client.get_run(PARENT_RUN_ID).info.experiment_id

# Find all 5 nested child runs (fold_1 to fold_5)
child_runs = client.search_runs(
    experiment_ids=[exp_id],
    filter_string=f"tags.mlflow.parentRunId = '{PARENT_RUN_ID}'",
    order_by=["attribute.start_time ASC"]
)


# Search all logged models in the experiment, then map them to their source run
logged_models = mlflow.search_logged_models(
    experiment_ids=[exp_id],
    output_format="list",
)

model_by_run = {}
for m in logged_models:
    if m.source_run_id:
        model_by_run[m.source_run_id] = m

lgb_models = []
for fold, child in enumerate(child_runs, 1):
    run_id = child.info.run_id
    lm = model_by_run.get(run_id)
    if lm is None:
        print(f"Fold {fold}: no logged model found for run {run_id}")
        continue
    model_uri = f"models:/{lm.model_id}"
    print(f"Loading fold {fold} from {model_uri}...")
    lgb_models.append(mlflow.lightgbm.load_model(model_uri))

print(f"Loaded {len(lgb_models)} models")

Loading fold 1 from models:/m-261c5ca54ab84a54a64b0207baa10786...


Loading fold 2 from models:/m-0ef31f24fad5484d833560f812386170...


Loading fold 3 from models:/m-b6f58430d46c4bdc9c5365f327a12ff6...


Loading fold 4 from models:/m-cb5140b74cc14dbab63af16c9a1d5c92...


Loading fold 5 from models:/m-e9811f26b2d54bcb9ef8204827289ca4...


Loaded 5 models


In [9]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
import json

LOCAL_LGBM_DIR = "models/lgbm_folds"
os.makedirs(LOCAL_LGBM_DIR, exist_ok=True)

df = pd.read_parquet(processed_dataset)

with open("artifacts/model_features.json", "r") as f:  
    features = json.load(f)

td_cols = df[features].select_dtypes(include=["timedelta", "datetime"]).columns.tolist()
for c in td_cols:
    df[c] = df[c].dt.total_seconds().astype("float32")

# 1. Setup TimeSeriesSplit (must match the training configuration)
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

oof_lgb = np.full(len(df), np.nan, dtype=np.float32)

# 2. Iterate through folds and generate validation predictions
for fold, ((train_idx, val_idx), model) in enumerate(zip(tscv.split(df), lgb_models), 1):
    print(f"Generating OOF predictions for Fold {fold}...")
    
    # Extract validation slice with the trained features
    X_val = df.iloc[val_idx][features]
    
    val_preds = model.predict(X_val)
        
    oof_lgb[val_idx] = val_preds

# 3. Save the reconstructed OOF array locally
np.save(os.path.join(LOCAL_LGBM_DIR, "oof_lgb.npy"), oof_lgb)
print(f"Reconstructed and saved {os.path.join(LOCAL_LGBM_DIR, 'oof_lgb.npy')} successfully!")

Generating OOF predictions for Fold 1...
Generating OOF predictions for Fold 2...
Generating OOF predictions for Fold 3...
Generating OOF predictions for Fold 4...
Generating OOF predictions for Fold 5...
Reconstructed and saved models/lgbm_folds\oof_lgb.npy successfully!


In [14]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score


# 1. Load DLRM OOF and Mask from your downloaded Drive folder
models_dir = src_path / "models"
dlrm_dir= models_dir / "dlrm_folds"
oof_dlrm = np.load(os.path.join(dlrm_dir, "oof_dlrm.npy"))
oof_mask = np.load(os.path.join(dlrm_dir, "oof_mask.npy"))

# 2. Filter strictly to rows evaluated during CV (where both have predictions)
eval_mask = oof_mask & ~np.isnan(oof_lgb)

y_true = df.loc[eval_mask, TARGET].values
lgb_eval = oof_lgb[eval_mask]
dlrm_eval = oof_dlrm[eval_mask]
n_eval = len(y_true)

# 3. Convert raw predictions to percentile ranks [0.0, 1.0]
lgb_rank = rankdata(lgb_eval) / n_eval
dlrm_rank = rankdata(dlrm_eval) / n_eval

# Standalone baselines
lgb_roc = roc_auc_score(y_true, lgb_eval)
lgb_pr = average_precision_score(y_true, lgb_eval)
dlrm_roc = roc_auc_score(y_true, dlrm_eval)
dlrm_pr = average_precision_score(y_true, dlrm_eval)

print("\n--- Standalone Baselines ---")
print(f"LightGBM OOF | ROC-AUC: {lgb_roc:.5f} | PR-AUC: {lgb_pr:.5f}")
print(f"DLRM     OOF | ROC-AUC: {dlrm_roc:.5f} | PR-AUC: {dlrm_pr:.5f}\n")

# 4. Grid search blend weight: w * LGB + (1 - w) * DLRM
weights = np.linspace(0.0, 1.0, 21)
best_pr = -1.0
best_w_lgb = 1.0
blend_records = []

for w in weights:
    w_lgb = round(w, 2)
    w_dlrm = round(1.0 - w_lgb, 2)
    
    blended_rank = (w_lgb * lgb_rank) + (w_dlrm * dlrm_rank)
    
    score_roc = roc_auc_score(y_true, blended_rank)
    score_pr = average_precision_score(y_true, blended_rank)
    
    blend_records.append({
        "w_lgb": w_lgb,
        "w_dlrm": w_dlrm,
        "roc_auc": score_roc,
        "pr_auc": score_pr,
        "delta_vs_lgb": score_roc - lgb_roc
    })
    
    if score_pr > best_pr:
        best_pr = score_pr
        best_w_lgb = w_lgb

results_df = pd.DataFrame(blend_records)
best_w_dlrm = round(1.0 - best_w_lgb, 2)

print("--- Blending Weight Search ---")
print(results_df.to_string(index=False))

print("\n" + "=" * 55)
print(f"Optimal Blend : {best_w_lgb:.2f} * LGB + {best_w_dlrm:.2f} * DLRM")
print(f"Best PR-AUC  : {best_pr:.5f} (Delta vs LGB: {best_pr - lgb_pr:+.5f})")
print("=" * 55)


--- Standalone Baselines ---
LightGBM OOF | ROC-AUC: 0.91670 | PR-AUC: 0.59896
DLRM     OOF | ROC-AUC: 0.87440 | PR-AUC: 0.53673

--- Blending Weight Search ---
 w_lgb  w_dlrm  roc_auc   pr_auc  delta_vs_lgb
  0.00    1.00 0.874401 0.536731     -0.042300
  0.05    0.95 0.879167 0.554874     -0.037534
  0.10    0.90 0.883546 0.564044     -0.033155
  0.15    0.85 0.887690 0.570549     -0.029011
  0.20    0.80 0.891647 0.575680     -0.025054
  0.25    0.75 0.895432 0.579922     -0.021269
  0.30    0.70 0.899038 0.583555     -0.017663
  0.35    0.65 0.902442 0.586720     -0.014259
  0.40    0.60 0.905611 0.589543     -0.011090
  0.45    0.55 0.908500 0.592085     -0.008201
  0.50    0.50 0.911065 0.594385     -0.005636
  0.55    0.45 0.913271 0.596456     -0.003430
  0.60    0.40 0.915122 0.598347     -0.001579
  0.65    0.35 0.916621 0.600055     -0.000081
  0.70    0.30 0.917777 0.601599      0.001075
  0.75    0.25 0.918602 0.602970      0.001901
  0.80    0.20 0.919091 0.604146      0